# Part C — Hyperparameter Tuning
Part B trained with default hyperparameters and overfit (train RMSE $3,857  vs test $8,443, a 2.2× gap). Here we run a SageMaker Automatic Model Tuning job to search for better hyperparameters, using a validation set and early stopping.

## Step 1 — Setup and Verify Data
Same session, role and bucket as Part B. Confirm the new train/validation files and the frozen test file are all readable before launching anything.

In [ ]:
import boto3, sagemaker 
from sagemaker import image_uris

BUCKET = "used-cars-ha"        # <-- same bucket as Part B
PREFIX = "data"

sess   = sagemaker.Session()
region = sess.boto_region_name
role   = sagemaker.get_execution_role()

print("region:", region)
print("role  :", role)

s3 = boto3.client("s3")
for f in ["train_c/train_c.csv", "validation_c/validation_c.csv", "test/test.csv"]:
    r = s3.head_object(Bucket=BUCKET, Key=f"{PREFIX}/{f}")
    print(f"OK  {f}  {r['ContentLength']:,} bytes")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


region: eu-north-1
role  : arn:aws:iam::481719141347:role/service-role/AmazonSageMaker-ExecutionRole-20260805T185648
OK  train_c/train_c.csv  900,803 bytes
OK  validation_c/validation_c.csv  225,292 bytes
OK  test/test.csv  283,069 bytes


## Step 2 — Base Estimator with Early Stopping
Same XGBoost container as Part B. Two changes: (1) num_round raised to 500 so early stopping has room to work, (2) early_stopping_rounds=20 — training halts when the validation score stops improving, which prevents the model memorising the training data.

In [2]:
from sagemaker.estimator import Estimator

xgb_image = image_uris.retrieve("xgboost", region, version="1.7-1")

est = Estimator(
    image_uri         = xgb_image,
    role              = role,
    instance_count    = 1,
    instance_type     = "ml.m5.large",
    output_path       = f"s3://{BUCKET}/model-tuned",
    sagemaker_session = sess,
    base_job_name     = "usedcar-xgb-tuned",
)

# fixed hyperparameters (the tuner will search the others in Step 3)
est.set_hyperparameters(
    objective            = "reg:squarederror",
    num_round            = 500,   # high ceiling; early stopping decides the real number
    early_stopping_rounds= 20,    # stop if validation RMSE doesn't improve for 20 rounds
)
print("base estimator ready")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


base estimator ready


## Step 3 — Define the Tuning Job
The tuner launches many training jobs, each with a different hyperparameter combination, and searches for the lowest validation RMSE. We search the four hyperparameters most responsible for overfitting.

In [3]:
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter

# the ranges the tuner is allowed to explore
hp_ranges = {
    "eta":              ContinuousParameter(0.01, 0.3),   # learning rate
    "max_depth":        IntegerParameter(3, 8),           # tree depth
    "subsample":        ContinuousParameter(0.5, 1.0),    # fraction of rows per tree
    "colsample_bytree": ContinuousParameter(0.5, 1.0),    # fraction of columns per tree
}

tuner = HyperparameterTuner(
    estimator             = est,
    objective_metric_name = "validation:rmse",   # what to minimise
    objective_type        = "Minimize",
    hyperparameter_ranges = hp_ranges,
    max_jobs              = 20,   # total combinations to try
    max_parallel_jobs     = 2,    # how many run at once
    strategy              = "Bayesian",
)
print("tuner configured: 20 jobs, 2 at a time")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/tuner.py:689: SageMakerV2DeprecationWarning: HyperparameterTuner is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use the hyperparameter tuning classes under sagemaker.core.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
HyperparameterTuner is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use the hyperparameter tuning classes under sagemaker.core.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


tuner configured: 20 jobs, 2 at a time


## Step 4 — Launch the Tuning Job
Wire in both channels — train_c and validation_c — and start the search. The tuner runs up to 20 training jobs; early stopping trims each one. Expect ~15-20 minutes.

In [4]:
from sagemaker.inputs import TrainingInput

train_input = TrainingInput(f"s3://{BUCKET}/{PREFIX}/train_c/",      content_type="text/csv")
val_input   = TrainingInput(f"s3://{BUCKET}/{PREFIX}/validation_c/", content_type="text/csv")

tuner.fit({"train": train_input, "validation": val_input})
tuner.wait()
print("tuning complete")

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


...........................................................................................................................................................................................................!
!
tuning complete


## Step 5 — Inspect the Best Hyperparameters
Pull the results of all 20 jobs, sorted by validation RMSE, and see which combination won.

In [5]:
tuning_job = tuner.latest_tuning_job.name
print("tuning job:", tuning_job, "\n")

# results of all 20 jobs as a dataframe
results = sagemaker.HyperparameterTuningJobAnalytics(tuning_job).dataframe()
results = results.sort_values("FinalObjectiveValue")   # lowest validation RMSE first

cols = ["eta", "max_depth", "subsample", "colsample_bytree",
        "FinalObjectiveValue", "TrainingJobStatus"]
print(results[cols].head(10).to_string(index=False))

best = tuner.best_estimator()
print("\nBest hyperparameters:")
for k in ["eta", "max_depth", "subsample", "colsample_bytree"]:
    print(f"  {k:18s}: {best.hyperparameters()[k]}")

tuning job: sagemaker-xgboost-260806-1415 

     eta  max_depth  subsample  colsample_bytree  FinalObjectiveValue TrainingJobStatus
0.116443        3.0   0.994130          0.803553          9029.135742         Completed
0.216880        6.0   0.998526          0.821598          9051.461914         Completed
0.084033        5.0   0.799077          0.766222          9061.791992         Completed
0.271004        5.0   0.996379          0.840277          9062.565430         Completed
0.293439        5.0   0.993534          0.817905          9082.374023         Completed
0.113691        5.0   0.994760          0.800387          9246.274414         Completed
0.183111        4.0   0.545442          0.854851          9269.494141         Completed
0.300000        3.0   1.000000          0.761746          9294.994141         Completed
0.071865        6.0   0.659979          0.500000          9330.346680         Completed
0.032112        8.0   0.578585          0.510412          9356.810547       

## Step 6 — Evaluate Best Model on the Frozen Test Set
Batch Transform the best model over the same test_features.csv used in Part B, so the comparison is apples-to-apples. Compute RMSE, MAE, R² and compare against the baseline model.

In [6]:
best = tuner.best_estimator()

transformer = best.transformer(
    instance_count = 1,
    instance_type  = "ml.m5.large",
    output_path    = f"s3://{BUCKET}/batch-out-tuned",
    assemble_with  = "Line",
    accept         = "text/csv",
)
transformer.transform(
    f"s3://{BUCKET}/{PREFIX}/inference/test_features.csv",
    content_type = "text/csv",
    split_type   = "Line",
)
transformer.wait()
print("predictions written to:", transformer.output_path)


2026-08-06 14:26:07 Starting - Found matching resource for reuse
2026-08-06 14:26:07 Downloading - Downloading the training image
2026-08-06 14:26:07 Training - Training image download completed. Training in progress.
2026-08-06 14:26:07 Uploading - Uploading generated training model
2026-08-06 14:26:07 Completed - Resource reused by training job: sagemaker-xgboost-260806-1415-012-78b4d7bb

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/transformer.py:115: SageMakerV2DeprecationWarning: Transformer is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.TransformJob`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Transformer is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.TransformJob`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-08-06-14-41-57-161


.........................../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-08-06:14:46:31:INFO] No GPUs detected (normal if no gpus installed)
[2026-08-06:14:46:31:INFO] No GPUs detected (normal if no gpus installed)
[2026-08-06:14:46:31:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocat

In [7]:
import numpy as np, pandas as pd

s3.download_file(BUCKET, "batch-out-tuned/test_features.csv.out", "preds_tuned.csv")
s3.download_file(BUCKET, f"{PREFIX}/test/test.csv", "test_local.csv")

y_pred = pd.read_csv("preds_tuned.csv", header=None)[0].values
y_true = pd.read_csv("test_local.csv",  header=None)[0].values
assert len(y_pred) == len(y_true)

err  = y_true - y_pred
rmse = np.sqrt(np.mean(err**2)); mae = np.mean(np.abs(err))
r2   = 1 - np.sum(err**2)/np.sum((y_true - y_true.mean())**2)

print("=== TUNED MODEL (test set) ===")
print(f"RMSE : ${rmse:,.0f}")
print(f"MAE  : ${mae:,.0f}")
print(f"R²   : {r2:.4f}")
print("\n=== vs PART B (default) ===")
print(f"Part B  RMSE $8,443 | MAE $5,799 | R² 0.8558")
print(f"Tuned   RMSE ${rmse:,.0f} | MAE ${mae:,.0f} | R² {r2:.4f}")

=== TUNED MODEL (test set) ===
RMSE : $8,247
MAE  : $5,630
R²   : 0.8625

=== vs PART B (default) ===
Part B  RMSE $8,443 | MAE $5,799 | R² 0.8558
Tuned   RMSE $8,247 | MAE $5,630 | R² 0.8625
